# Advanced Bi-GRU with CNN + Attention for Protein Secondary Structure

**Target: F1 Harmonic ≥ 0.60**

## Architecture Improvements:
- Multi-scale CNN for local feature extraction
- Self-attention for long-range dependencies
- Residual connections + Layer normalization
- Class weighting for Q8 imbalance

In [ ]:
!pip install -q trackio

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math

class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Model architecture - Enhanced
    embed_dim = 256  # Increased from 128
    cnn_channels = 128
    hidden_dim = 512
    num_layers = 3
    num_heads = 8  # For attention
    dropout = 0.4  # Increased from 0.35
    
    # Training
    batch_size = 64
    learning_rate = 5e-4  # Reduced from 1e-3
    weight_decay = 1e-4  # Increased from 3e-5
    epochs = 60
    grad_clip = 0.5  # Reduced from 1.0
    label_smoothing = 0.1
    
    # Data split
    val_split = 0.1
    
    # Paths
    train_path = 'train.csv'
    test_path = 'test.csv'
    model_save_path = 'best_model_bigru_v9.pth'
    submission_path = 'submission_bigru_v9.csv'
    
    # TrackIO
    trackio_project = '25-t3-nppe2'
    trackio_space_id = 'manyakaistha/dlgenai-nppe'
    trackio_group = 'bi_gru_v9_advanced'

cfg = Config()
print(f'Device: {cfg.device}')

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

In [ ]:
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'hf_placeholder_token_for_reproducibility'
os.environ['TRACKIO_SPACE_ID'] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        token = os.environ.get('HUGGINGFACEHUB_API_TOKEN')
        if cfg.trackio_space_id and token:
            trackio.init(project=cfg.trackio_project, space_id=cfg.trackio_space_id, group=cfg.trackio_group)
            print('✓ TrackIO initialized')
            return trackio
    except Exception as e:
        print(f'TrackIO init failed: {e}')
    return None

trk = maybe_init_trackio()

In [ ]:
PAD_TOKEN = 0

AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

print(f'Vocab: AA={len(AA_TO_ID)}, Q8={len(Q8_TO_ID)}, Q3={len(Q3_TO_ID)}')

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, data, mode='train'):
        self.data = data
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        seq_ids = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in row['seq']]
        item = {'seq': torch.tensor(seq_ids, dtype=torch.long), 'id': row['id']}
        if self.mode == 'train':
            item['q8'] = torch.tensor([Q8_TO_ID[c] for c in row['sst8']], dtype=torch.long)
            item['q3'] = torch.tensor([Q3_TO_ID[c] for c in row['sst3']], dtype=torch.long)
        return item

def collate_fn(batch):
    seqs = pad_sequence([item['seq'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
    out = {'seq': seqs, 'ids': [item['id'] for item in batch], 'lengths': torch.tensor([len(item['seq']) for item in batch])}
    if 'q8' in batch[0]:
        out['q8'] = pad_sequence([item['q8'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
        out['q3'] = pad_sequence([item['q3'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
    return out

df_train_full = pd.read_csv(cfg.train_path)
train_data, val_data = train_test_split(df_train_full, test_size=cfg.val_split, random_state=cfg.seed)

train_ds = ProteinDataset(train_data.reset_index(drop=True), mode='train')
val_ds = ProteinDataset(val_data.reset_index(drop=True), mode='train')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
def compute_class_weights(labels_column, vocab):
    """Compute inverse frequency weights for class balancing."""
    counts = {c: 0 for c in vocab if c != '<PAD>'}
    for label_str in labels_column:
        for c in label_str:
            if c in counts:
                counts[c] += 1
    total = sum(counts.values())
    weights = torch.ones(len(vocab))
    for c, cnt in counts.items():
        if cnt > 0:
            idx = vocab.index(c)
            weights[idx] = total / (len(counts) * cnt)
    weights[0] = 0.0  # PAD token
    return weights / weights[1:].max()  # Normalize

q8_weights = compute_class_weights(train_data['sst8'], Q8_CHARS).to(cfg.device)
q3_weights = compute_class_weights(train_data['sst3'], Q3_CHARS).to(cfg.device)
print(f'Q8 weights: {q8_weights}')
print(f'Q3 weights: {q3_weights}')

In [ ]:
class MultiScaleCNN(nn.Module):
    """Multi-scale 1D CNN for local feature extraction."""
    def __init__(self, in_dim, out_dim, kernels=[3, 5, 7], dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(in_dim, out_dim, k, padding=k//2) for k in kernels
        ])
        self.proj = nn.Linear(out_dim * len(kernels), out_dim * 2)
        self.norm = nn.LayerNorm(out_dim * 2)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: [B, L, D]
        x = x.transpose(1, 2)  # [B, D, L]
        conv_outs = [torch.relu(conv(x)) for conv in self.convs]
        x = torch.cat(conv_outs, dim=1)  # [B, out*3, L]
        x = x.transpose(1, 2)  # [B, L, out*3]
        x = self.proj(x)
        return self.dropout(self.norm(x))


class SelfAttention(nn.Module):
    """Multi-head self-attention."""
    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        attn_out, _ = self.attn(x, x, x, key_padding_mask=mask)
        return self.norm(x + self.dropout(attn_out))


class AdvancedBiGRU(nn.Module):
    """CNN + BiGRU + Attention for protein secondary structure prediction."""
    
    def __init__(self, vocab_size, embed_dim, cnn_channels, hidden_dim, num_layers,
                 num_heads, num_q8, num_q3, dropout=0.4):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_TOKEN)
        
        # Multi-scale CNN
        self.cnn = MultiScaleCNN(embed_dim, cnn_channels, dropout=dropout)
        
        # BiGRU
        gru_input_dim = cnn_channels * 2
        self.gru = nn.GRU(gru_input_dim, hidden_dim, num_layers, batch_first=True,
                         bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        
        self.gru_norm = nn.LayerNorm(hidden_dim * 2)
        
        # Projection for residual (match dimensions)
        self.residual_proj = nn.Linear(gru_input_dim, hidden_dim * 2)
        
        # Self-attention
        self.attention = SelfAttention(hidden_dim * 2, num_heads, dropout)
        
        self.dropout = nn.Dropout(dropout)
        
        # Classification heads with intermediate layers
        self.q8_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_q8)
        )
        
        self.q3_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_q3)
        )
    
    def forward(self, x):
        # x: [B, L]
        pad_mask = (x == PAD_TOKEN)
        
        embed = self.embedding(x)  # [B, L, embed_dim]
        
        # CNN features
        cnn_out = self.cnn(embed)  # [B, L, cnn_channels*2]
        
        # GRU
        gru_out, _ = self.gru(cnn_out)  # [B, L, hidden*2]
        
        # Residual connection
        residual = self.residual_proj(cnn_out)
        gru_out = self.gru_norm(gru_out + residual)
        
        # Self-attention with residual
        attn_out = self.attention(gru_out, pad_mask)
        
        out = self.dropout(attn_out)
        
        q8_logits = self.q8_head(out)
        q3_logits = self.q3_head(out)
        
        return q8_logits, q3_logits


model = AdvancedBiGRU(
    vocab_size=len(AA_TO_ID),
    embed_dim=cfg.embed_dim,
    cnn_channels=cfg.cnn_channels,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    num_heads=cfg.num_heads,
    num_q8=len(Q8_TO_ID),
    num_q3=len(Q3_TO_ID),
    dropout=cfg.dropout
).to(cfg.device)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion_q8 = nn.CrossEntropyLoss(weight=q8_weights, ignore_index=PAD_TOKEN, label_smoothing=cfg.label_smoothing)
criterion_q3 = nn.CrossEntropyLoss(weight=q3_weights, ignore_index=PAD_TOKEN, label_smoothing=cfg.label_smoothing)

optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

class ModelCheckpoint:
    def __init__(self, filepath):
        self.filepath = filepath
        self.best_score = -np.inf

    def __call__(self, score, model, optimizer, epoch, metrics):
        if score > self.best_score:
            print(f'✓ Saving best model (epoch {epoch+1}, score={score:.4f})')
            self.best_score = score
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                       'optimizer_state_dict': optimizer.state_dict(),
                       'best_score': score, 'metrics': metrics}, self.filepath)
            return True
        return False

checkpoint = ModelCheckpoint(cfg.model_save_path)
print('Training setup complete!')

In [ ]:
def calculate_f1(preds, targets, pad_idx=PAD_TOKEN):
    preds_flat = preds.argmax(dim=-1).view(-1).cpu().numpy()
    targets_flat = targets.view(-1).cpu().numpy()
    mask = targets_flat != pad_idx
    return f1_score(targets_flat[mask], preds_flat[mask], average='macro', zero_division=0)

def harmonic_mean(a, b):
    return 2 * (a * b) / (a + b) if (a + b) > 0 else 0

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device, epoch):
    model.train()
    total_loss, total_q8_f1, total_q3_f1, count = 0, 0, 0, 0
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} Train')
    for batch in pbar:
        seq = batch['seq'].to(device)
        q8_tgt = batch['q8'].to(device)
        q3_tgt = batch['q3'].to(device)
        
        optimizer.zero_grad()
        q8_logits, q3_logits = model(seq)
        
        loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
        loss = loss_q8 + loss_q3
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step(epoch + count / len(loader))
        
        total_loss += loss.item()
        total_q8_f1 += calculate_f1(q8_logits.detach(), q8_tgt)
        total_q3_f1 += calculate_f1(q3_logits.detach(), q3_tgt)
        count += 1
        
        pbar.set_postfix({'loss': loss.item(), 'q8': total_q8_f1/count, 'q3': total_q3_f1/count})
        
        if trk:
            trk.log({'train/loss': loss.item(), 'train/f1_q8': total_q8_f1/count, 'train/f1_q3': total_q3_f1/count})
    
    return total_loss/count, total_q8_f1/count, total_q3_f1/count, harmonic_mean(total_q8_f1/count, total_q3_f1/count)


def validate(model, loader, device):
    model.eval()
    total_loss = 0
    all_q8_preds, all_q8_targets, all_q3_preds, all_q3_targets = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            seq = batch['seq'].to(device)
            q8_tgt = batch['q8'].to(device)
            q3_tgt = batch['q3'].to(device)
            
            q8_logits, q3_logits = model(seq)
            
            loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            total_loss += (loss_q8 + loss_q3).item()
            
            mask = q8_tgt != PAD_TOKEN
            all_q8_preds.extend(q8_logits.argmax(-1)[mask].cpu().numpy())
            all_q8_targets.extend(q8_tgt[mask].cpu().numpy())
            all_q3_preds.extend(q3_logits.argmax(-1)[mask].cpu().numpy())
            all_q3_targets.extend(q3_tgt[mask].cpu().numpy())
    
    f1_q8 = f1_score(all_q8_targets, all_q8_preds, average='macro', zero_division=0)
    f1_q3 = f1_score(all_q3_targets, all_q3_preds, average='macro', zero_division=0)
    return total_loss/len(loader), f1_q8, f1_q3, harmonic_mean(f1_q8, f1_q3)

In [ ]:
history = {'train_loss': [], 'train_f1_q8': [], 'train_f1_q3': [], 'train_f1_harm': [],
           'val_loss': [], 'val_f1_q8': [], 'val_f1_q3': [], 'val_f1_harm': []}
best_val_f1 = 0

print('Starting training...')
print(f'Target: F1 Harmonic ≥ 0.60\n')

for epoch in range(cfg.epochs):
    train_loss, train_q8, train_q3, train_harm = train_epoch(model, train_loader, optimizer, scheduler, cfg.device, epoch)
    val_loss, val_q8, val_q3, val_harm = validate(model, val_loader, cfg.device)
    
    for k, v in [('train_loss', train_loss), ('train_f1_q8', train_q8), ('train_f1_q3', train_q3), ('train_f1_harm', train_harm),
                 ('val_loss', val_loss), ('val_f1_q8', val_q8), ('val_f1_q3', val_q3), ('val_f1_harm', val_harm)]:
        history[k].append(v)
    
    print(f'\nEpoch {epoch+1}/{cfg.epochs}:')
    print(f'  Train Loss: {train_loss:.4f} | Q8: {train_q8:.4f} | Q3: {train_q3:.4f} | Harm: {train_harm:.4f}')
    print(f'  Val   Loss: {val_loss:.4f} | Q8: {val_q8:.4f} | Q3: {val_q3:.4f} | Harm: {val_harm:.4f}')
    
    checkpoint(val_harm, model, optimizer, epoch, {'val_q8': val_q8, 'val_q3': val_q3, 'val_harm': val_harm})
    if val_harm > best_val_f1:
        best_val_f1 = val_harm
    
    if trk:
        trk.log({'epoch': epoch+1, 'val/loss': val_loss, 'val/f1_q8': val_q8, 'val/f1_q3': val_q3, 'val/f1_harm': val_harm})
    
    print('-' * 60)

print(f'\n✓ Training complete! Best Val F1 Harmonic: {best_val_f1:.4f}')
print(f'Target achieved: {"✓ YES" if best_val_f1 >= 0.60 else "✗ NO"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_f1_harm'], label='Train Harm')
axes[1].plot(history['val_f1_harm'], label='Val Harm')
axes[1].plot(history['val_f1_q8'], '--', label='Val Q8')
axes[1].plot(history['val_f1_q3'], '--', label='Val Q3')
axes[1].axhline(0.60, color='r', linestyle=':', label='Target')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].set_title('F1 Scores')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves_v9.png', dpi=150)
plt.show()

In [ ]:
print(f'Loading best model from: {cfg.model_save_path}')
ckpt = torch.load(cfg.model_save_path, map_location=cfg.device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'✓ Loaded from epoch {ckpt["epoch"]+1}, score: {ckpt["best_score"]:.4f}')

df_test = pd.read_csv(cfg.test_path)
test_ds = ProteinDataset(df_test, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

model.eval()
results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Inference'):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['lengths']
        q8_logits, q3_logits = model(seq)
        q8_preds = q8_logits.argmax(-1).cpu().numpy()
        q3_preds = q3_logits.argmax(-1).cpu().numpy()
        
        for i, id_ in enumerate(batch['ids']):
            L = lengths[i].item()
            results.append({
                'id': id_,
                'sst8': ''.join([ID_TO_Q8[idx] for idx in q8_preds[i, :L]]),
                'sst3': ''.join([ID_TO_Q3[idx] for idx in q3_preds[i, :L]])
            })

df_sub = pd.DataFrame(results)
df_sub.to_csv(cfg.submission_path, index=False)
print(f'\n✓ Submission saved: {cfg.submission_path}')
print(df_sub.head())

In [ ]:
print('Validating submission...')
assert len(df_sub) == len(df_test)
valid_q8 = set('BCEGIHST')
valid_q3 = set('CEH')
for idx, row in df_sub.iterrows():
    seq_len = len(df_test.loc[df_test['id'] == row['id'], 'seq'].values[0])
    assert len(row['sst8']) == seq_len and len(row['sst3']) == seq_len
    assert all(c in valid_q8 for c in row['sst8']) and all(c in valid_q3 for c in row['sst3'])
print('✓ Submission valid!')